In [1]:
from pathlib import Path
import pandas as pd
import re

In [2]:
base_path = Path("../data/bronze/laps/season=2026")

In [4]:
files = sorted(base_path.rglob("*.parquet"))
print(f"Found {len(files)} files")
for f in files:
    print(f"- {f}")

Found 3 files
- ..\data\bronze\laps\season=2026\round=01\laps.parquet
- ..\data\bronze\laps\season=2026\round=02\laps.parquet
- ..\data\bronze\laps\season=2026\round=03\laps.parquet


In [5]:
dfs = [pd.read_parquet(f) for f in files]
df = pd.concat(dfs, ignore_index=True)

In [6]:
df_silver = df.drop(columns=[
    "FastF1Generated", 
    "ingested_at_utc", 
    "IsAccurate", 
    "Sector1SessionTime", 
    "Sector2SessionTime", 
    "Sector3SessionTime",
    "LapStartTime",
    "LapStartDate",
    "Deleted",
    "DeletedReason"
])

In [7]:
key_cols = ["season", "round_number", "Driver", "LapNumber"]
dup_cnt = df_silver.duplicated(subset=key_cols).sum()
print(f"Duplicated by key: {dup_cnt}")

df_silver.drop

Duplicated by key: 0


<bound method DataFrame.drop of                        Time Driver DriverNumber                LapTime  \
0    0 days 01:03:56.437000    NOR            1 0 days 00:01:36.458000   
1    0 days 01:05:23.781000    NOR            1 0 days 00:01:27.344000   
2    0 days 01:06:50.644000    NOR            1 0 days 00:01:26.863000   
3    0 days 01:08:16.501000    NOR            1 0 days 00:01:25.857000   
4    0 days 01:09:42.074000    NOR            1 0 days 00:01:25.573000   
...                     ...    ...          ...                    ...   
3033 0 days 01:31:41.771000    BEA           87 0 days 00:01:57.448000   
3034 0 days 01:33:17.825000    BEA           87 0 days 00:01:36.054000   
3035 0 days 01:34:53.730000    BEA           87 0 days 00:01:35.905000   
3036 0 days 01:36:29.334000    BEA           87 0 days 00:01:35.604000   
3037 0 days 01:38:59.334000    BEA           87                    NaT   

      LapNumber  Stint             PitOutTime PitInTime  \
0           1.0    1

In [288]:
df.to_csv("./cek_isi.csv", index=False)

**FUNCTION**

In [289]:
def check_null(column):
    amount = column.isna().sum()
    print(f"Null data for {column.name}: {amount}")
    if amount > 0:
        mask = column.isna()
        if mask.any():
            print(f"Rows with null in {column.name}:")
            display(df_silver.loc[mask])
def strip_string(column):
    print(f"Stripping string for {column.name}")
    column = column.str.strip()
    
    return column

def upper_string(column):
    print(f"Uppercasing string for {column.name}")
    column = column.str.upper()

    return column

def clean_driver(column):
    column = strip_string(column)
    column = upper_string(column)

    return column

def convert_to_int(column):
    print(f"Converting to int for {column.name}")
    column = pd.to_numeric(column, errors="coerce")
    
    return column


**TIME**

In [290]:
check_null(df_silver["Time"])

Null data for Time: 0


**DRIVER**

In [291]:
check_null(df_silver["Driver"])
df_silver.value_counts("Driver")
df_silver["Driver"] = clean_driver(df_silver["Driver"])

Null data for Driver: 0
Stripping string for Driver
Uppercasing string for Driver


**DRIVER NUMBER**

In [292]:
check_null(df_silver["DriverNumber"])
df_silver["DriverNumber"] = convert_to_int(df_silver["DriverNumber"])

Null data for DriverNumber: 0
Converting to int for DriverNumber


**LAP TIME**

In [293]:
check_null(df_silver["LapTime"])
df_silver["is_LapTime_not_na"] = df_silver["LapTime"].notna()

Null data for LapTime: 48
Rows with null in LapTime:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Compound,TyreLife,FreshTyre,Team,TrackStatus,Position,season,round_number,event_name,session_type
240,0 days 01:38:59.952000,ALO,14,NaT,13.0,2.0,NaT,0 days 01:22:49.657000,0 days 00:00:35.885000,0 days 00:00:25.378000,...,SOFT,2.0,True,Aston Martin,6712,19.0,2026,1,Australian Grand Prix,R
249,0 days 01:52:27.116000,ALO,14,NaT,22.0,3.0,NaT,0 days 01:52:27.205000,0 days 00:00:30.824000,0 days 00:00:30.521000,...,SOFT,11.0,False,Aston Martin,167,18.0,2026,1,Australian Grand Prix,R
341,0 days 02:13:18.247000,STR,18,NaT,34.0,3.0,NaT,0 days 01:55:18.611000,0 days 00:00:30.496000,0 days 00:00:19.613000,...,SOFT,8.0,True,Aston Martin,1,17.0,2026,1,Australian Grand Prix,R
408,0 days 01:03:51.908000,HUL,27,NaT,1.0,1.0,NaT,NaT,NaT,NaT,...,HARD,1.0,True,Audi,1,NaN,2026,1,Australian Grand Prix,R
875,0 days 01:19:15.504000,HAD,6,NaT,11.0,1.0,NaT,NaT,NaT,NaT,...,MEDIUM,11.0,True,Red Bull Racing,16,NaN,2026,1,Australian Grand Prix,R
945,0 days 01:22:05.068000,BOT,77,NaT,12.0,1.0,NaT,0 days 01:21:36.868000,0 days 00:00:46.720000,0 days 00:00:45.988000,...,HARD,12.0,True,Cadillac,67,19.0,2026,1,Australian Grand Prix,R
949,0 days 01:29:21.898000,BOT,77,NaT,16.0,2.0,NaT,NaT,NaT,NaT,...,HARD,16.0,False,Cadillac,126,NaN,2026,1,Australian Grand Prix,R
1007,0 days 00:58:03.760000,NOR,1,NaT,1.0,1.0,NaT,NaT,NaT,NaT,...,HARD,1.0,True,McLaren,12,NaN,2026,2,Chinese Grand Prix,R
1018,0 days 01:15:47.651000,GAS,10,NaT,11.0,2.0,0 days 01:13:30.471000,NaT,0 days 00:00:54.680000,0 days 00:00:40.781000,...,HARD,1.0,True,Alpine,4,9.0,2026,2,Chinese Grand Prix,R
1019,0 days 01:18:17.634000,GAS,10,NaT,12.0,2.0,NaT,NaT,0 days 00:00:42.739000,0 days 00:00:44.215000,...,HARD,2.0,True,Alpine,4,9.0,2026,2,Chinese Grand Prix,R


**LAP NUMBER**

In [294]:
df_silver["LapNumber"] = convert_to_int(df_silver["LapNumber"]).astype(int)
check_null(df_silver["LapNumber"])

Converting to int for LapNumber
Null data for LapNumber: 0


**STINT**

In [295]:
df_silver["Stint"] = convert_to_int(df_silver["Stint"]).astype(int)
check_null(df_silver["Stint"])

Converting to int for Stint
Null data for Stint: 0


**PIT OUT TIME**

In [296]:
check_null(df_silver["PitOutTime"])
df_silver["is_pit_out_time_not_na"] = df_silver["PitOutTime"].notna()

Null data for PitOutTime: 2958
Rows with null in PitOutTime:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,TyreLife,FreshTyre,Team,TrackStatus,Position,season,round_number,event_name,session_type,is_LapTime_not_na
0,0 days 01:03:56.437000,NOR,1,0 days 00:01:36.458000,1,1,NaT,NaT,NaT,0 days 00:00:18.163000,...,1.0,True,McLaren,1,6.0,2026,1,Australian Grand Prix,R,True
1,0 days 01:05:23.781000,NOR,1,0 days 00:01:27.344000,2,1,NaT,NaT,0 days 00:00:31.074000,0 days 00:00:18.116000,...,2.0,True,McLaren,1,6.0,2026,1,Australian Grand Prix,R,True
2,0 days 01:06:50.644000,NOR,1,0 days 00:01:26.863000,3,1,NaT,NaT,0 days 00:00:30.541000,0 days 00:00:18.252000,...,3.0,True,McLaren,1,7.0,2026,1,Australian Grand Prix,R,True
3,0 days 01:08:16.501000,NOR,1,0 days 00:01:25.857000,4,1,NaT,NaT,0 days 00:00:30.190000,0 days 00:00:18.193000,...,4.0,True,McLaren,1,7.0,2026,1,Australian Grand Prix,R,True
4,0 days 01:09:42.074000,NOR,1,0 days 00:01:25.573000,5,1,NaT,NaT,0 days 00:00:29.930000,0 days 00:00:17.868000,...,5.0,True,McLaren,1,7.0,2026,1,Australian Grand Prix,R,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3032,0 days 01:29:44.323000,BEA,87,0 days 00:01:38.787000,16,1,NaT,0 days 01:29:43.059000,0 days 00:00:35.528000,0 days 00:00:42.582000,...,16.0,True,Haas F1 Team,1,16.0,2026,3,Japanese Grand Prix,R,True
3034,0 days 01:33:17.825000,BEA,87,0 days 00:01:36.054000,18,2,NaT,NaT,0 days 00:00:35.006000,0 days 00:00:42.424000,...,2.0,True,Haas F1 Team,1,19.0,2026,3,Japanese Grand Prix,R,True
3035,0 days 01:34:53.730000,BEA,87,0 days 00:01:35.905000,19,2,NaT,NaT,0 days 00:00:35.345000,0 days 00:00:42.252000,...,3.0,True,Haas F1 Team,1,18.0,2026,3,Japanese Grand Prix,R,True
3036,0 days 01:36:29.334000,BEA,87,0 days 00:01:35.604000,20,2,NaT,NaT,0 days 00:00:35.253000,0 days 00:00:42.310000,...,4.0,True,Haas F1 Team,1,18.0,2026,3,Japanese Grand Prix,R,True


**PIT IN TIME**

In [297]:
check_null(df_silver["PitInTime"])
df_silver["is_pit_in_time_not_na"] = df_silver["PitInTime"].notna()

Null data for PitInTime: 2954
Rows with null in PitInTime:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,TrackStatus,Position,season,round_number,event_name,session_type,is_LapTime_not_na,is_pit_out_time_not_na
0,0 days 01:03:56.437000,NOR,1,0 days 00:01:36.458000,1,1,NaT,NaT,NaT,0 days 00:00:18.163000,...,True,McLaren,1,6.0,2026,1,Australian Grand Prix,R,True,False
1,0 days 01:05:23.781000,NOR,1,0 days 00:01:27.344000,2,1,NaT,NaT,0 days 00:00:31.074000,0 days 00:00:18.116000,...,True,McLaren,1,6.0,2026,1,Australian Grand Prix,R,True,False
2,0 days 01:06:50.644000,NOR,1,0 days 00:01:26.863000,3,1,NaT,NaT,0 days 00:00:30.541000,0 days 00:00:18.252000,...,True,McLaren,1,7.0,2026,1,Australian Grand Prix,R,True,False
3,0 days 01:08:16.501000,NOR,1,0 days 00:01:25.857000,4,1,NaT,NaT,0 days 00:00:30.190000,0 days 00:00:18.193000,...,True,McLaren,1,7.0,2026,1,Australian Grand Prix,R,True,False
4,0 days 01:09:42.074000,NOR,1,0 days 00:01:25.573000,5,1,NaT,NaT,0 days 00:00:29.930000,0 days 00:00:17.868000,...,True,McLaren,1,7.0,2026,1,Australian Grand Prix,R,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3033,0 days 01:31:41.771000,BEA,87,0 days 00:01:57.448000,17,2,0 days 01:30:08.107000,NaT,0 days 00:00:57.139000,0 days 00:00:41.840000,...,True,Haas F1 Team,1,19.0,2026,3,Japanese Grand Prix,R,True,True
3034,0 days 01:33:17.825000,BEA,87,0 days 00:01:36.054000,18,2,NaT,NaT,0 days 00:00:35.006000,0 days 00:00:42.424000,...,True,Haas F1 Team,1,19.0,2026,3,Japanese Grand Prix,R,True,False
3035,0 days 01:34:53.730000,BEA,87,0 days 00:01:35.905000,19,2,NaT,NaT,0 days 00:00:35.345000,0 days 00:00:42.252000,...,True,Haas F1 Team,1,18.0,2026,3,Japanese Grand Prix,R,True,False
3036,0 days 01:36:29.334000,BEA,87,0 days 00:01:35.604000,20,2,NaT,NaT,0 days 00:00:35.253000,0 days 00:00:42.310000,...,True,Haas F1 Team,1,18.0,2026,3,Japanese Grand Prix,R,True,False


**SECTORS TIME**

In [298]:
df_silver["is_s1_notna"] = df_silver["Sector1Time"].notna()
df_silver["is_s2_notna"] = df_silver["Sector2Time"].notna()
df_silver["is_s3_notna"] = df_silver["Sector3Time"].notna()
df_silver["is_sector_complete"] = (
    df_silver["is_s1_notna"]
    & df_silver["is_s2_notna"]
    & df_silver["is_s3_notna"]
)

**SPEED**

In [299]:
df_silver["SpeedI1"] = convert_to_int(df_silver["SpeedI1"])
df_silver["SpeedI2"] = convert_to_int(df_silver["SpeedI2"])
df_silver["SpeedST"] = convert_to_int(df_silver["SpeedST"])
df_silver["SpeedFL"] = convert_to_int(df_silver["SpeedFL"])

check_null(df_silver["SpeedI1"])
check_null(df_silver["SpeedI2"])
check_null(df_silver["SpeedST"])
check_null(df_silver["SpeedFL"])

speed_cols = ["SpeedI1", "SpeedI2", "SpeedST", "SpeedFL"]
df_silver["is_speed_complete"] = df_silver[speed_cols].notna().all(axis=1)


Converting to int for SpeedI1
Converting to int for SpeedI2
Converting to int for SpeedST
Converting to int for SpeedFL
Null data for SpeedI1: 411
Rows with null in SpeedI1:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,round_number,event_name,session_type,is_LapTime_not_na,is_pit_out_time_not_na,is_pit_in_time_not_na,is_s1_notna,is_s2_notna,is_s3_notna,is_sector_complete
7,0 days 01:13:57.387000,NOR,1,0 days 00:01:25.299000,8,1,NaT,NaT,0 days 00:00:30.191000,0 days 00:00:17.970000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
86,0 days 01:45:51.909000,GAS,10,0 days 00:01:25.096000,29,2,NaT,NaT,0 days 00:00:29.838000,0 days 00:00:18.010000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
89,0 days 01:50:07.624000,GAS,10,0 days 00:01:25.202000,32,2,NaT,NaT,0 days 00:00:30.200000,0 days 00:00:17.946000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
125,0 days 01:18:49.564000,PER,11,0 days 00:01:37.875000,11,1,NaT,NaT,0 days 00:00:31.102000,0 days 00:00:18.422000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
135,0 days 01:35:14.410000,PER,11,0 days 00:01:27.262000,21,2,NaT,NaT,0 days 00:00:30.520000,0 days 00:00:18.341000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3008,0 days 02:19:35.795000,PIA,81,0 days 00:01:33.523000,45,2,NaT,NaT,0 days 00:00:34.114000,0 days 00:00:41.348000,...,3,Japanese Grand Prix,R,True,False,False,True,True,True,True
3026,0 days 01:20:00.585000,BEA,87,0 days 00:01:36.764000,10,1,NaT,NaT,0 days 00:00:35.806000,0 days 00:00:42.842000,...,3,Japanese Grand Prix,R,True,False,False,True,True,True,True
3031,0 days 01:28:05.536000,BEA,87,0 days 00:01:36.624000,15,1,NaT,NaT,0 days 00:00:35.622000,0 days 00:00:42.581000,...,3,Japanese Grand Prix,R,True,False,False,True,True,True,True
3032,0 days 01:29:44.323000,BEA,87,0 days 00:01:38.787000,16,1,NaT,0 days 01:29:43.059000,0 days 00:00:35.528000,0 days 00:00:42.582000,...,3,Japanese Grand Prix,R,True,False,True,True,True,True,True


Null data for SpeedI2: 9
Rows with null in SpeedI2:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,round_number,event_name,session_type,is_LapTime_not_na,is_pit_out_time_not_na,is_pit_in_time_not_na,is_s1_notna,is_s2_notna,is_s3_notna,is_sector_complete
408,0 days 01:03:51.908000,HUL,27,NaT,1,1,NaT,NaT,NaT,NaT,...,1,Australian Grand Prix,R,False,False,False,False,False,False,False
875,0 days 01:19:15.504000,HAD,6,NaT,11,1,NaT,NaT,NaT,NaT,...,1,Australian Grand Prix,R,False,False,False,False,False,False,False
949,0 days 01:29:21.898000,BOT,77,NaT,16,2,NaT,NaT,NaT,NaT,...,1,Australian Grand Prix,R,False,False,False,False,False,False,False
1007,0 days 00:58:03.760000,NOR,1,NaT,1,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
1272,0 days 01:14:14.241000,STR,18,NaT,10,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
1273,0 days 00:58:03.760000,ALB,23,NaT,1,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
1651,0 days 00:58:03.760000,BOR,5,NaT,1,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
1874,0 days 00:58:03.760000,PIA,81,NaT,1,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
3037,0 days 01:38:59.334000,BEA,87,NaT,21,2,NaT,NaT,NaT,NaT,...,3,Japanese Grand Prix,R,False,False,False,False,False,False,False


Null data for SpeedST: 74
Rows with null in SpeedST:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,round_number,event_name,session_type,is_LapTime_not_na,is_pit_out_time_not_na,is_pit_in_time_not_na,is_s1_notna,is_s2_notna,is_s3_notna,is_sector_complete
4,0 days 01:09:42.074000,NOR,1,0 days 00:01:25.573000,5,1,NaT,NaT,0 days 00:00:29.930000,0 days 00:00:17.868000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
28,0 days 01:45:28.455000,NOR,1,0 days 00:01:24.080000,29,2,NaT,NaT,0 days 00:00:29.692000,0 days 00:00:17.992000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
29,0 days 01:46:52.416000,NOR,1,0 days 00:01:23.961000,30,2,NaT,NaT,0 days 00:00:29.515000,0 days 00:00:17.966000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
47,0 days 02:12:30.246000,NOR,1,0 days 00:01:22.953000,48,3,NaT,NaT,0 days 00:00:29.106000,0 days 00:00:17.743000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
53,0 days 02:20:46.349000,NOR,1,0 days 00:01:22.796000,54,3,NaT,NaT,0 days 00:00:29.006000,0 days 00:00:17.779000,...,1,Australian Grand Prix,R,True,False,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1272,0 days 01:14:14.241000,STR,18,NaT,10,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
1273,0 days 00:58:03.760000,ALB,23,NaT,1,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
1651,0 days 00:58:03.760000,BOR,5,NaT,1,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False
1874,0 days 00:58:03.760000,PIA,81,NaT,1,1,NaT,NaT,NaT,NaT,...,2,Chinese Grand Prix,R,False,False,False,False,False,False,False


Null data for SpeedFL: 93
Rows with null in SpeedFL:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,round_number,event_name,session_type,is_LapTime_not_na,is_pit_out_time_not_na,is_pit_in_time_not_na,is_s1_notna,is_s2_notna,is_s3_notna,is_sector_complete
10,0 days 01:18:31.712000,NOR,1,0 days 00:01:43.391000,11,1,NaT,0 days 01:18:15.593000,0 days 00:00:30.117000,0 days 00:00:17.987000,...,1,Australian Grand Prix,R,True,False,True,True,True,True,True
33,0 days 01:52:58.107000,NOR,1,0 days 00:01:52.732000,34,2,NaT,0 days 01:52:42.584000,0 days 00:00:39.736000,0 days 00:00:22.345000,...,1,Australian Grand Prix,R,True,False,True,True,True,True,True
68,0 days 01:18:37.508000,GAS,10,0 days 00:01:44.821000,11,1,NaT,0 days 01:18:21.012000,0 days 00:00:30.082000,0 days 00:00:17.797000,...,1,Australian Grand Prix,R,True,False,True,True,True,True,True
132,0 days 01:30:37.299000,PER,11,0 days 00:02:06.084000,18,1,NaT,0 days 01:30:18.038000,0 days 00:00:35.121000,0 days 00:00:24.446000,...,1,Australian Grand Prix,R,True,False,True,True,True,True,True
157,0 days 02:08:04.625000,PER,11,0 days 00:01:45.049000,43,2,NaT,0 days 02:07:47.788000,0 days 00:00:31.257000,0 days 00:00:18.463000,...,1,Australian Grand Prix,R,True,False,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2879,0 days 01:37:04.163000,RUS,63,0 days 00:01:36.505000,21,1,NaT,0 days 01:37:02.807000,0 days 00:00:34.583000,0 days 00:00:41.632000,...,3,Japanese Grand Prix,R,True,False,True,True,True,True,True
2930,0 days 01:35:17.633000,BOT,77,0 days 00:01:41.943000,19,1,NaT,0 days 01:35:16.312000,0 days 00:00:36.686000,0 days 00:00:43.598000,...,3,Japanese Grand Prix,R,True,False,True,True,True,True,True
2981,0 days 01:32:18.857000,PIA,81,0 days 00:01:37.327000,18,1,NaT,0 days 01:32:17.551000,0 days 00:00:34.808000,0 days 00:00:41.961000,...,3,Japanese Grand Prix,R,True,False,True,True,True,True,True
3032,0 days 01:29:44.323000,BEA,87,0 days 00:01:38.787000,16,1,NaT,0 days 01:29:43.059000,0 days 00:00:35.528000,0 days 00:00:42.582000,...,3,Japanese Grand Prix,R,True,False,True,True,True,True,True


**IS PERSONAL BEST**

In [300]:
df_silver["IsPersonalBest"] = df_silver["IsPersonalBest"].astype("boolean")
check_null(df_silver["IsPersonalBest"])

Null data for IsPersonalBest: 4
Rows with null in IsPersonalBest:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,event_name,session_type,is_LapTime_not_na,is_pit_out_time_not_na,is_pit_in_time_not_na,is_s1_notna,is_s2_notna,is_s3_notna,is_sector_complete,is_speed_complete
875,0 days 01:19:15.504000,HAD,6,NaT,11,1,NaT,NaT,NaT,NaT,...,Australian Grand Prix,R,False,False,False,False,False,False,False,False
949,0 days 01:29:21.898000,BOT,77,NaT,16,2,NaT,NaT,NaT,NaT,...,Australian Grand Prix,R,False,False,False,False,False,False,False,False
1272,0 days 01:14:14.241000,STR,18,NaT,10,1,NaT,NaT,NaT,NaT,...,Chinese Grand Prix,R,False,False,False,False,False,False,False,False
3037,0 days 01:38:59.334000,BEA,87,NaT,21,2,NaT,NaT,NaT,NaT,...,Japanese Grand Prix,R,False,False,False,False,False,False,False,False


**COMPOUND**

In [301]:
df_silver["Compound"] = strip_string(df_silver["Compound"])
df_silver["Compound"] = upper_string(df_silver["Compound"])
df_silver.groupby("Compound").size()

Stripping string for Compound
Uppercasing string for Compound


Compound
HARD      1931
MEDIUM     953
SOFT       154
dtype: int64

**TYRE LIFE**

In [302]:
check_null(df_silver["TyreLife"])
df_silver["TyreLife"] = convert_to_int(df_silver["TyreLife"]).astype(int)


Null data for TyreLife: 0
Converting to int for TyreLife


**FRESH TYRE**

In [303]:
check_null(df_silver["FreshTyre"])
df_silver.groupby("FreshTyre").size()

Null data for FreshTyre: 0


FreshTyre
False      63
True     2975
dtype: int64

**TEAM**

In [304]:
check_null(df_silver["Team"])
df_silver["Team"] = strip_string(df_silver["Team"])
df_silver.groupby("Team").size()

Null data for Team: 0
Stripping string for Team


Team
Alpine             330
Aston Martin       189
Audi               220
Cadillac           286
Ferrari            334
Haas F1 Team       299
McLaren            166
Mercedes           334
Racing Bulls       331
Red Bull Racing    276
Williams           273
dtype: int64

**TRACK STATUS**

In [305]:
check_null(df_silver["TrackStatus"])
df_silver["TrackStatus"] = convert_to_int(df_silver["TrackStatus"]).astype(int)
df_silver["is_green_flag"] = df_silver["TrackStatus"].eq(1)


Null data for TrackStatus: 0
Converting to int for TrackStatus


**POSITION**

In [306]:
check_null(df_silver["Position"])
df_silver["is_position_not_na"] = df_silver["Position"].notna()

Null data for Position: 9
Rows with null in Position:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,session_type,is_LapTime_not_na,is_pit_out_time_not_na,is_pit_in_time_not_na,is_s1_notna,is_s2_notna,is_s3_notna,is_sector_complete,is_speed_complete,is_green_flag
408,0 days 01:03:51.908000,HUL,27,NaT,1,1,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,True
875,0 days 01:19:15.504000,HAD,6,NaT,11,1,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False
949,0 days 01:29:21.898000,BOT,77,NaT,16,2,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False
1007,0 days 00:58:03.760000,NOR,1,NaT,1,1,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False
1272,0 days 01:14:14.241000,STR,18,NaT,10,1,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False
1273,0 days 00:58:03.760000,ALB,23,NaT,1,1,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False
1651,0 days 00:58:03.760000,BOR,5,NaT,1,1,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False
1874,0 days 00:58:03.760000,PIA,81,NaT,1,1,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False
3037,0 days 01:38:59.334000,BEA,87,NaT,21,2,NaT,NaT,NaT,NaT,...,R,False,False,False,False,False,False,False,False,False


**SEASON**

In [307]:
check_null(df_silver["season"])
df_silver.groupby("season").size()

Null data for season: 0


season
2026    3038
dtype: int64

**ROUND NUMBER**

In [308]:
check_null(df_silver["round_number"])
df_silver.groupby("round_number").size()

Null data for round_number: 0


round_number
1    1007
2     924
3    1107
dtype: int64

**EVENT NAME**

In [309]:
check_null(df_silver["event_name"])
df_silver["event_name"] = strip_string(df_silver["event_name"])
df_silver["event_name"] = upper_string(df_silver["event_name"])
df_silver.groupby("event_name").size()

Null data for event_name: 0
Stripping string for event_name
Uppercasing string for event_name


event_name
AUSTRALIAN GRAND PRIX    1007
CHINESE GRAND PRIX        924
JAPANESE GRAND PRIX      1107
dtype: int64

**SESSION TYPE**

In [310]:
check_null(df_silver["session_type"])
df_silver["session_type"] = strip_string(df_silver["session_type"])
df_silver["session_type"] = upper_string(df_silver["session_type"])
df_silver.groupby("session_type").size()

Null data for session_type: 0
Stripping string for session_type
Uppercasing string for session_type


session_type
R    3038
dtype: int64

In [311]:
df_silver.info()
df_silver

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype          
---  ------                  --------------  -----          
 0   Time                    3038 non-null   timedelta64[ns]
 1   Driver                  3038 non-null   object         
 2   DriverNumber            3038 non-null   int64          
 3   LapTime                 2990 non-null   timedelta64[ns]
 4   LapNumber               3038 non-null   int64          
 5   Stint                   3038 non-null   int64          
 6   PitOutTime              80 non-null     timedelta64[ns]
 7   PitInTime               84 non-null     timedelta64[ns]
 8   Sector1Time             2968 non-null   timedelta64[ns]
 9   Sector2Time             3029 non-null   timedelta64[ns]
 10  Sector3Time             3026 non-null   timedelta64[ns]
 11  SpeedI1                 2627 non-null   float64        
 12  SpeedI2                 3029 non-n

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,is_LapTime_not_na,is_pit_out_time_not_na,is_pit_in_time_not_na,is_s1_notna,is_s2_notna,is_s3_notna,is_sector_complete,is_speed_complete,is_green_flag,is_position_not_na
0,0 days 01:03:56.437000,NOR,1,0 days 00:01:36.458000,1,1,NaT,NaT,NaT,0 days 00:00:18.163000,...,True,False,False,False,True,True,False,True,True,True
1,0 days 01:05:23.781000,NOR,1,0 days 00:01:27.344000,2,1,NaT,NaT,0 days 00:00:31.074000,0 days 00:00:18.116000,...,True,False,False,True,True,True,True,True,True,True
2,0 days 01:06:50.644000,NOR,1,0 days 00:01:26.863000,3,1,NaT,NaT,0 days 00:00:30.541000,0 days 00:00:18.252000,...,True,False,False,True,True,True,True,True,True,True
3,0 days 01:08:16.501000,NOR,1,0 days 00:01:25.857000,4,1,NaT,NaT,0 days 00:00:30.190000,0 days 00:00:18.193000,...,True,False,False,True,True,True,True,True,True,True
4,0 days 01:09:42.074000,NOR,1,0 days 00:01:25.573000,5,1,NaT,NaT,0 days 00:00:29.930000,0 days 00:00:17.868000,...,True,False,False,True,True,True,True,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3033,0 days 01:31:41.771000,BEA,87,0 days 00:01:57.448000,17,2,0 days 01:30:08.107000,NaT,0 days 00:00:57.139000,0 days 00:00:41.840000,...,True,True,False,True,True,True,True,True,True,True
3034,0 days 01:33:17.825000,BEA,87,0 days 00:01:36.054000,18,2,NaT,NaT,0 days 00:00:35.006000,0 days 00:00:42.424000,...,True,False,False,True,True,True,True,True,True,True
3035,0 days 01:34:53.730000,BEA,87,0 days 00:01:35.905000,19,2,NaT,NaT,0 days 00:00:35.345000,0 days 00:00:42.252000,...,True,False,False,True,True,True,True,True,True,True
3036,0 days 01:36:29.334000,BEA,87,0 days 00:01:35.604000,20,2,NaT,NaT,0 days 00:00:35.253000,0 days 00:00:42.310000,...,True,False,False,True,True,True,True,True,True,True


In [312]:
df_silver.to_csv("../data/silver/done_silver.csv", index=False)